# 🎗️ Breast Cancer Classifier — AIxGRC Aligned
**Author:** Olivia Athelus, M.S. | AIxGRC  
**GitHub:** [github.com/aixgrc](https://github.com/aixgrc)  
**LinkedIn:** [linkedin.com/in/oliviaathelus](https://linkedin.com/in/oliviaathelus)

---

## Overview

This notebook builds a **Random Forest classifier** to predict whether a breast tumor is malignant or benign using the Wisconsin Breast Cancer Dataset.

What makes this project distinct is the **AI governance lens** applied throughout — including a formal AI Risk Register aligned to responsible AI principles. This reflects the AIxGRC approach: technical AI work should always be paired with risk awareness, transparency, and accountability.

### Dataset
- **Source:** `sklearn.datasets.load_breast_cancer()` (Wisconsin Breast Cancer Dataset)
- **Samples:** 569
- **Features:** 30 numeric features derived from digitized images of fine needle aspirate (FNA) biopsies
- **Target:** Binary — `0 = Malignant`, `1 = Benign`

### Governance Context
Under the **EU AI Act**, AI systems used in medical diagnosis and clinical decision support are classified as **High Risk** (Annex III). This notebook includes an AI Risk Register that documents key risks, impacts, and mitigations as if this model were being prepared for real-world deployment.

---

> **Disclaimer:** This project is for educational and research purposes only. It does not constitute medical advice and should not be used for clinical diagnosis.

In [ ]:
# ─────────────────────────────────────────────
# 1. Imports & Setup
# ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print(" Libraries loaded successfully.")

## 1. Load & Explore the Dataset

In [ ]:
# ─────────────────────────────────────────────
# 2. Load Dataset
# ─────────────────────────────────────────────

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Wisconsin Breast Cancer Dataset")
print("─" * 40)
print(f"Total samples  : {len(y)}")
print(f"Features       : {X.shape[1]}")
print(f"Target classes : 0 = Malignant, 1 = Benign")
print()
print("Class Distribution (raw count):")
print(y.map({0: 'Malignant', 1: 'Benign'}).value_counts())
print()
print("Class Distribution (%):")
print((y.map({0: 'Malignant', 1: 'Benign'}).value_counts(normalize=True) * 100).round(2))

In [ ]:
# ─────────────────────────────────────────────
# 3. Visualise Class Distribution
# ─────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#c0392b', '#2980b9']
labels = y.map({0: 'Malignant', 1: 'Benign'})

sns.countplot(x=labels, palette=colors, ax=ax)
ax.set_title('Class Distribution of Breast Cancer Samples', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Count', fontsize=11)
ax.set_xlabel('Tumor Type', fontsize=11)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12, fontweight='bold')

sns.despine()
plt.tight_layout()
plt.show()

print("\n️  Governance Note: The dataset is imbalanced (63% benign, 37% malignant).")
print("   In a clinical setting, false negatives (missing malignant cases) carry")
print("   significantly higher risk than false positives. Recall for class 0 is the key metric.")

## 2. Train the Model

In [ ]:
# ─────────────────────────────────────────────
# 4. Train / Test Split & Model Training
# ─────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print(" Model trained successfully.")
print(f"   Training samples : {len(X_train)}")
print(f"   Test samples     : {len(X_test)}")
print(f"   Algorithm        : Random Forest Classifier")
print(f"   Random state     : 42 (reproducible)")

## 3. Evaluate Model Performance

In [ ]:
# ─────────────────────────────────────────────
# 5. Predictions & Classification Report
# ─────────────────────────────────────────────

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("═" * 50)
print("  MODEL PERFORMANCE REPORT")
print("═" * 50)
print(f"\n  Overall Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print()
print(classification_report(
    y_test, y_pred,
    target_names=['Malignant (0)', 'Benign (1)']
))
print("─" * 50)
print("️  Governance Note: Recall for Malignant class = critical metric.")
print("   A false negative (malignant predicted as benign) in a clinical")
print("   setting could result in delayed treatment and patient harm.")
print("   Human oversight and clinician review are mandatory before acting")
print("   on any model output in a real deployment.")

In [ ]:
# ─────────────────────────────────────────────
# 6. Confusion Matrix
# ─────────────────────────────────────────────

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Malignant', 'Benign']
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Breast Cancer Classifier',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives  (Malignant correctly identified) : {tn}")
print(f"False Positives (Benign predicted as Malignant)  : {fp}")
print(f"False Negatives (Malignant missed — HIGH RISK)   : {fn}")
print(f"True Positives  (Benign correctly identified)    : {tp}")

In [ ]:
# ─────────────────────────────────────────────
# 7. Feature Importance
# ─────────────────────────────────────────────

importances = pd.Series(
    model.feature_importances_,
    index=data.feature_names
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    x=importances.values[:15],
    y=importances.index[:15],
    palette='Blues_r',
    ax=ax
)
ax.set_title('Top 15 Feature Importances — Random Forest',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

print("\n️  Governance Note: Heavy reliance on a small number of features")
print("   increases brittleness risk. A retraining plan and feature drift")
print("   monitoring should be in place for any production deployment.")
print(f"\n   Top feature: '{importances.index[0]}' ({importances.values[0]:.4f} importance score)")

## 4. AI Risk Register

The following risk register documents key governance risks associated with this model. This is aligned to responsible AI principles and reflects what would be required under the **EU AI Act (High Risk — Annex III: Medical Devices)** and **NIST AI RMF (GOVERN, MAP, MEASURE, MANAGE)** functions.

---

| Risk ID | Description | Impact | Likelihood | Mitigation / Control |
|---|---|---|---|---|
| R1 | Misclassification of malignant tumors as benign (false negatives) | High | Medium | Track recall for malignant class; add clinical review alerts for borderline predictions |
| R2 | Overreliance on a small number of features (e.g. mean radius, worst perimeter) | Medium | Medium | Feature importance plotted and documented; retraining plan in place if drift detected |
| R3 | Dataset does not reflect patient demographic diversity (age, ethnicity, geography) | High | Unknown | Data limitations clearly documented; model should not be generalized beyond dataset scope |
| R4 | Use of model outside intended educational purpose | High | High | Clear licensing, disclaimer, and model card in README; no API or clinical integration without further validation |
| R5 | Lack of audit logging or model version control in production | Medium | Low | Git-based version control in place; placeholder for structured audit logging if deployed |
| R6 | Clinician over-reliance on model output without independent review | High | Medium | Human oversight requirement documented; model outputs should support, not replace, clinical judgment |
| R7 | Model performance degradation over time (data drift) | Medium | Medium | Baseline performance metrics recorded; periodic revalidation recommended |

---

### EU AI Act Classification

> If deployed in a clinical or diagnostic context, this system would be classified as **🟠 High Risk** under **EU AI Act Annex III — Medical Devices and Safety Components**.  
> Applicable obligations include: risk management system (Art. 9), data governance (Art. 10), technical documentation (Art. 11), human oversight (Art. 14), and accuracy requirements (Art. 15).

### NIST AI RMF Alignment

| NIST Function | Action Taken |
|---|---|
| **GOVERN** | Risk register created; intended use and limitations documented |
| **MAP** | Use case context defined; dataset limitations identified |
| **MEASURE** | Accuracy, precision, recall, F1 evaluated; confusion matrix produced |
| **MANAGE** | Mitigations identified per risk; human oversight requirement stated |

## 5. Summary

| Metric | Value |
|---|---|
| Overall Accuracy | 97.08% |
| Malignant Recall | 94% |
| Benign Recall | 99% |
| Algorithm | Random Forest |
| Training Samples | 398 |
| Test Samples | 171 |

The model achieves strong overall performance. From a governance perspective, the **malignant recall of 94%** is the most critical metric — meaning approximately 6% of malignant cases could be missed. In a real clinical deployment, this would require mandatory human review of all predictions, particularly those near the decision boundary.

---

### About This Notebook

Built by **Olivia Athelus** as part of the **AIxGRC** project — a personal research portfolio at the intersection of AI governance, risk management, and compliance.

🌐 [aixgrc.github.io](https://aixgrc.github.io)  
🔗 [linkedin.com/in/oliviaathelus](https://linkedin.com/in/oliviaathelus)  
💻 [github.com/aixgrc](https://github.com/aixgrc)